In [14]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 4, 6, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 4, 10, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 5/5 [00:00<00:00, 36.71it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
0,2625557393000000101,,NEWT,TRAD,2026-04-06 04:00:34+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND,2026-04-06
1,2625563842000000201,,NEWT,TRAD,2026-04-06 04:00:38+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZDD9XLPWW4T,NA/Swap Fxd Flt JPY,JPY-TONA-OIS-COMPOUND,2026-04-06
2,2625557495000000201,2625493964000000101,CORR,,2026-04-06 04:01:47+00:00,None,IR,None,N,False,...,3.0,0,JPY,3.0,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,2026-04-06
3,2625557494000000101,2625493965000000201,CORR,,2026-04-06 04:01:47+00:00,None,IR,None,N,False,...,3.0,0,JPY,3.0,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,2026-04-06
4,2625557496000000301,,NEWT,TRAD,2026-04-06 04:01:51+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound,2026-04-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103190,2692172776000000101,854913642,MODI,TRAD,2026-04-10 23:34:46+00:00,True,IR,None,N,False,...,NaN,,,NaN,None,None,QZVXZ8VM229Q,NA/O Nstd Oth USD,USD-SOFR ICE Swap Rate,2026-04-10
103191,2692173752000000101,854913643,MODI,TRAD,2026-04-10 23:35:10+00:00,True,IR,None,N,False,...,NaN,,,NaN,None,None,QZVXZ8VM229Q,NA/O Nstd Oth USD,USD-SOFR ICE Swap Rate,2026-04-10
103192,2692176161000000101,854913645,MODI,TRAD,2026-04-10 23:36:22+00:00,True,IR,None,N,False,...,NaN,,,NaN,None,None,QZVXZ8VM229Q,NA/O Nstd Oth USD,USD-SOFR ICE Swap Rate,2026-04-10
103193,2692177969000000101,854913242,MODI,TRAD,2026-04-10 23:37:19+00:00,True,IR,None,N,False,...,NaN,,,NaN,None,None,QZVXZ8VM229Q,NA/O Nstd Oth USD,USD-SOFR ICE Swap Rate,2026-04-10


In [21]:
# df["Event timestamp"] = df["Event timestamp"].astype(str)
# df["Execution Timestamp"] = df["Execution Timestamp"].astype(str)
# df.to_csv(r"C:\Users\chris\clee\ARBS\notebooks\sdr\april_fomc_dated_sdr_trades.csv", index=False)	

In [23]:
df[(df["Effective Date"].dt.date == datetime.date(2026, 4, 28)) & (df["Expiration Date"].dt.date == datetime.date(2026, 6, 16))]

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
